# 🇧🇩 Bangla Transactional Document KIE: QLoRA Fine-Tuning & Evaluation
### End-to-End Instruction Tuning on Qwen2.5-3B with Verifier-Grounded Evaluation

This notebook trains and evaluates a lightweight, cost-effective LLM (**Qwen2.5-3B-Instruct**) for **Bangla Key Information Extraction (KIE)** across utility bills, bKash/Nagad mobile receipts, and supermarket POS invoices with **strict span grounding**.

## 1. Setup & Environment Installation

In [7]:
!nvidia-smi
!pip install -q transformers peft trl bitsandbytes accelerate datasets jsonschema rapidfuzz matplotlib

Sun Oct 18 06:47:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Load Training Dataset & System Prompt

In [3]:
import glob
import os
import json
from datasets import load_dataset

# 1. Automatically find the file anywhere under /kaggle/input
files = glob.glob("/kaggle/input/**/sft_train_full.jsonl", recursive=True)

if not files:
    print("❌ Could not find sft_train_full.jsonl! Files currently in /kaggle/input:")
    for root, dirs, filenames in os.walk("/kaggle/input"):
        for f in filenames:
            print("  ", os.path.join(root, f))
else:
    TRAIN_FILE = files[0]
    print(f"✅ Found training file at: {TRAIN_FILE}")

    SYSTEM_PROMPT = """You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.
If a field is missing, omit it or set it to null. Never hallucinate values not present in the OCR text."""

    dataset = load_dataset("json", data_files=TRAIN_FILE)
    print(f"🎉 Successfully loaded {len(dataset['train'])} training examples!")
    print("\nSample preview:")
    print(dataset['train'][0]['messages'][1]['content'][:300], "...")


✅ Found training file at: /kaggle/input/datasets/isfakiqbalchowdhuruy/bangla-kie-dataset/data/sft_train_full.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

🎉 Successfully loaded 3000 training examples!

Sample preview:
### Document Type: mfs_receipt

### Target JSON Schema:
{
  "title": "BanglaMFSReceiptSchema",
  "description": "Schema for extracting span-grounded key information from Bangla Mobile Financial Services receipts (bKash, Nagad, Rocket, Upay, Cellfin).",
  "type": "object",
  "properties": {
    "docu ...


## 3. Load Model in 4-bit NormalFloat (NF4) with LoRA Adapters

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,             # Explicitly force float16
    device_map="auto",
    trust_remote_code=True
)

# Convert any stray bfloat16 tensors to float16
for param in model.parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## 4. Run SFT Training with TRL SFTTrainer

In [4]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "/kaggle/working/output/qwen2.5_3b_bangla_kie"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=800,           # Matches exact document size (3x speedup!)
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    num_train_epochs=1,       # 1 epoch is ~375 steps, perfect for SFT
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    optim="paged_adamw_8bit",
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    args=sft_config
)

print("🚀 Starting fast training on Kaggle GPU...")
trainer.train()

trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"🎉 Training complete! Model adapters saved to: {OUTPUT_DIR}")


Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 Starting fast training on Kaggle GPU...


Step,Training Loss
10,1.588663
20,0.845034
30,0.107700
40,0.006372
50,0.002709
60,0.002380
70,0.002285
80,0.002265
90,0.002137
100,0.002136


🎉 Training complete! Model adapters saved to: /kaggle/working/output/qwen2.5_3b_bangla_kie


## 5. Verify & Evaluate Against Grounded Verifier

In [5]:
# Run evaluation using the project's verifier module
!python eval/evaluate_predictions.py --test-file data/test/synthetic_test_300.json --predictions results/baselines/predictions_sft_round1.json

python3: can't open file '/kaggle/working/eval/evaluate_predictions.py': [Errno 2] No such file or directory


In [7]:
import torch

# 1. Put the already-trained model in evaluation mode
model.eval()

# 2. Test with a sample Bangla document
sample_ocr_text = """
বিকাশ স্টেটমেন্ট
লেনদেন সফল হয়েছে
ট্রানজেকশন আইডি: 9K8L2M5P01
তারিখ: ১৮/১০/২০২৬
প্রেরক: ০১৭xxxxxxxx
প্রাপক: ০১৮xxxxxxxx
পরিমাণ: ১,৫০০.০০ টাকা
চার্জ: ০.০০ টাকা
মোট: ১,৫০০.০০ টাকা
ধন্যবাদ
"""

prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.
If a field is missing, omit it or set it to null. Never hallucinate values not present in the OCR text.<|im_end|>
<|im_start|>user
Document Type: mfs_receipt
OCR Text:
{sample_ocr_text}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== Extracted Grounded JSON ===")
print(response)


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== Extracted Grounded JSON ===
{
  "from_mfs_transfer_in": {
    "value": "9K8L2M5P01",
    "source_span": "ট্রানজেকশন আইডি: 9K8L2M5P01"
  },
  "mfs_transaction_type": {
    "value": "লেনদেন সফল হয়েছে",
    "source_span": "লেনদেন সফল হয়েছে"
  },
  "transaction_id": {
    "value": "9K8L2M5P01",
    "source_span": "ট্রানজেকশন আইডি: 9K8L2M5P01"
  },
  "date": {
    "value": "2026-10-18",
    "source_span": "তারিখ: ১৮/১০/২০২৬"
  },
  "sender_account_number": {
    "value": "০১৭xxxxxxxx",
    "source_span": "প্রেরক: ০১৭xxxxxxxx"
  },
  "recipient_account_number": {
    "value": "০১৮xxxxxxxx",
    "source_span": "প্রাপক: ০১৮xxxxxxxx"
  },
  "amount_transfered": {
    "value": "১,৫০০.০০ টাকা",
    "source_span": "প্রাপক: ০১৮xxxxxxxx\nপ্রেরক: ১,৫০০.০০ টাকা"
  },
  "charge_fee": {
    "value": "০.০০ টাকা",
    "source_span": "চার্জ: ০.০০ টাকা"
  },
  "total_amount": {
    "value": "১,৫০০.০০ টাকা",
    "source_span": "মোট: ১,৫০০.


In [8]:
!zip -r /kaggle/working/bangla_kie_qwen3b_adapter.zip /kaggle/working/output/qwen2.5_3b_bangla_kie
print("✅ Created bangla_kie_qwen3b_adapter.zip! You can now download it from the right-hand Output panel.")


  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/ (stored 0%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/README.md (deflated 44%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/tokenizer_config.json (deflated 59%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/ (stored 0%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/README.md (deflated 65%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/tokenizer_config.json (deflated 59%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/rng_state.pth (deflated 26%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/training_args.bin (deflated 53%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/adapter_model.safetensors (deflated 22%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/optimizer.pt (deflated 12%)
  adding: kaggle/working/output/qwen2.5_3b_bangla_kie/checkpoint-188/train

In [9]:
# 1. Test Base Model (BEFORE training)
with model.disable_adapter():
    with torch.no_grad():
        out_base = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    base_response = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# 2. Test Fine-Tuned Model (AFTER training)
with torch.no_grad():
    out_tuned = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
tuned_response = tokenizer.decode(out_tuned[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=" * 25 + " BEFORE TRAINING (Base Model) " + "=" * 25)
print(base_response[:400])

print("\n" + "=" * 25 + " AFTER TRAINING (Your Fine-Tuned Model) " + "=" * 25)
print(tuned_response[:400])


========================= BEFORE TRAINING (Base Model) =========================
[
    {
        "value": "বিকাশ স্টেটমেন্ট",
        "source_span": "বিকাশ স্টেটমেন্ট"
    },
    {
        "value": "লেনদেন সফল হয়েছে",
        "source_span": "লেনদেন সফল হয়েছে"
    },
    {
        "value": "9K8L2M5P01",
        "source_span": "ট্রানজেকশন আইডি: 9K8L2M5P01"
    },
    {
        "value": "১৮/১০/২০২৬",
        "source_span": "তারিখ: ১৮/১০/২০২৬"
    },
    {
        "value": "০১৭x

========================= AFTER TRAINING (Your Fine-Tuned Model) =========================
{
  "from_mfs_transfer_in": {
    "value": "9K8L2M5P01",
    "source_span": "ট্রানজেকশন আইডি: 9K8L2M5P01"
  },
  "mfs_transaction_type": {
    "value": "লেনদেন সফল হয়েছে",
    "source_span": "লেনদেন সফল হয়েছে"
  },
  "transaction_id": {
    "value": "9K8L2M5P01",
    "source_span": "ট্রানজেকশন আইডি: 9K8L2M5P01"
  },
  "date": {
    "value": "2026-10-18",
    "source_span": "তারিখ: ১৮/১০/২০২৬"
  


In [4]:
import json
import re
import torch

TEST_FILE = "/kaggle/input/bangla-kie-dataset/data/synthetic_test_300.json"
with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_docs = json.load(f)[:15]  # Test on 15 unseen receipts

print("🔍 Running Head-to-Head Test: Base Model vs. Your Fine-Tuned Model...\n")

scores = {
    "Base_Model": {"json_valid": 0, "has_source_span": 0, "total_fields": 0},
    "Your_Model": {"json_valid": 0, "has_source_span": 0, "total_fields": 0}
}

for i, doc in enumerate(test_docs):
    prompt = f"""<|im_start|>system
You are an expert Bangla Document Key Information Extraction system.
Extract information strictly from the provided OCR text into the requested JSON schema.
For every extracted field, output an object with two keys:
1. 'value': the extracted or normalized data.
2. 'source_span': the exact substring copied character-for-character from the OCR text.<|im_end|>
<|im_start|>user
Document Type: {doc['document_type']}
OCR Text:
{doc['ocr_text']}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    # 1. Run BASE MODEL (Adapter DISABLED)
    with model.disable_adapter():
        with torch.no_grad():
            out_base = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        base_text = tokenizer.decode(out_base[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # 2. Run YOUR MODEL (Adapter ACTIVE)
    with torch.no_grad():
        out_tuned = model.generate(**inputs, max_new_tokens=400, temperature=0.1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    tuned_text = tokenizer.decode(out_tuned[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    # Parse and count
    for name, text in [("Base_Model", base_text), ("Your_Model", tuned_text)]:
        try:
            match = re.search(r"\{.*\}", text, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                scores[name]["json_valid"] += 1
                for k, v in data.items():
                    scores[name]["total_fields"] += 1
                    if isinstance(v, dict) and "source_span" in v:
                        scores[name]["has_source_span"] += 1
        except:
            pass

print("=" * 60)
print(f"{'Metric':<30} | {'Base Model (Previous)':<20} | {'Your Model (Now)':<20}")
print("=" * 60)
print(f"{'Clean JSON Syntax Rate':<30} | {scores['Base_Model']['json_valid']/15:<20.1%} | {scores['Your_Model']['json_valid']/15:<20.1%}")

base_span = scores['Base_Model']['has_source_span'] / max(1, scores['Base_Model']['total_fields'])
tuned_span = scores['Your_Model']['has_source_span'] / max(1, scores['Your_Model']['total_fields'])
print(f"{'Grounded Span Compliance':<30} | {base_span:<20.1%} | {tuned_span:<20.1%}")
print("=" * 60)

if tuned_span > base_span:
    print("\n✅ RESULT: Your fine-tuned model is significantly BETTER than previous!")
else:
    print("\n⚖️ RESULT: The models performed similarly.")


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/bangla-kie-dataset/data/synthetic_test_300.json'

In [6]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [8]:
import glob
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# 1. Automatically find your uploaded adapter
adapter_dirs = glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
if not adapter_dirs:
    # If it was uploaded as a zip
    zip_files = glob.glob("/kaggle/input/**/bangla_kie_qwen3b_adapter.zip", recursive=True)
    if zip_files:
        !unzip -o {zip_files[0]} -d /kaggle/working/adapter/
        adapter_dirs = glob.glob("/kaggle/working/**/adapter_model.safetensors", recursive=True)

import os
ADAPTER_PATH = os.path.dirname(adapter_dirs[0])
print(f"✅ Found and loading your trained adapter from: {ADAPTER_PATH}")

# 2. Load Base Model in 4-bit
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)

# 3. Attach your trained LoRA weights
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("🎉 Successfully loaded your fine-tuned model without re-training!")


✅ Found and loading your trained adapter from: /kaggle/input/datasets/isfakiqbalchowdhuruy/my-bangla-adapter/kaggle/working/output/qwen2.5_3b_bangla_kie


ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`